# OCTO Mobile Review - Regex Analysis

**Nama:** Alisha Rafimalia  
**NRP:** 5026231202  
**Kelas:** PBA A  
**Notebook:** 3 - Regex Analysis

Notebook ini fokus pada analisis berbasis Regular Expression (Regex) untuk memahami pola teks review pengguna: noise text, pola kontak, isu spesifik, dan dampak cleaning terhadap kualitas data NLP.

In [ ]:
!pip -q install pandas numpy seaborn matplotlib

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 11

print("Libraries loaded successfully.")

In [ ]:
# Load data dari beberapa kemungkinan lokasi
candidate_paths = [
    "../WEEK 2/hasil_scraping_octo_clean.csv",
    "WEEK 2/hasil_scraping_octo_clean.csv",
    "../.qodo/WEEK 2/hasil_scraping_octo_clean.csv",
    "./.qodo/WEEK 2/hasil_scraping_octo_clean.csv",
    ".qodo/WEEK 2/hasil_scraping_octo_clean.csv",
    "hasil_scraping_octo_clean.csv",
]

df = None
used_path = None

for p in candidate_paths:
    try:
        df = pd.read_csv(p)
        used_path = p
        break
    except FileNotFoundError:
        continue

if df is None:
    matches = list(Path.cwd().rglob("hasil_scraping_octo_clean.csv"))
    if matches:
        used_path = str(matches[0])
        df = pd.read_csv(used_path)

if df is None:
    raise FileNotFoundError("File hasil_scraping_octo_clean.csv tidak ditemukan.")

print(f"Data loaded from: {used_path}")
print("Shape:", df.shape)
df.head()

### Penjelasan Output
- Output path menunjukkan sumber file yang dipakai.
- Shape menunjukkan jumlah baris dan kolom data.
- Preview data dipakai untuk validasi bahwa kolom teks review siap dianalisis regex.

## 1. Regex Pattern Definition dan Detection

Mendefinisikan pola-pola teks penting yang ingin dideteksi dari review pengguna.

In [ ]:
regex_patterns = {
    "has_url": r"https?://\S+|www\.\S+",
    "has_email": r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
    "has_phone": r"(?:\+62|62|0)\d{8,13}",
    "has_number": r"\b\d+\b",
    "has_emoji": r"[\U0001F300-\U0001FAFF\u2600-\u27BF]",
    "has_repeated_char": r"(.)\1{2,}",
    "has_allcaps_word": r"\b[A-Z]{3,}\b",
}

df_regex = df.copy()
df_regex["content"] = df_regex["content"].fillna("").astype(str)

for col, pattern in regex_patterns.items():
    df_regex[col] = df_regex["content"].str.contains(pattern, regex=True)

prevalence = (df_regex[list(regex_patterns.keys())].mean() * 100).sort_values(ascending=False).round(2)
prevalence_df = prevalence.reset_index()
prevalence_df.columns = ["pattern", "pct_review"]
prevalence_df

In [ ]:
plt.figure(figsize=(10, 4))
sns.barplot(data=prevalence_df, x="pct_review", y="pattern", palette="mako")
plt.title("Persentase Review yang Mengandung Pola Regex")
plt.xlabel("Persentase Review (%)")
plt.ylabel("Pattern")
plt.tight_layout()
plt.show()

### Penjelasan Output
- Tabel dan grafik menunjukkan seberapa sering pola tertentu muncul di review.
- Pola dengan persentase tinggi mengindikasikan noise/karakteristik teks yang dominan.
- Hasil ini dipakai untuk menentukan aturan cleaning regex prioritas.

## 2. Regex-based Text Cleaning

Membersihkan teks review menggunakan aturan regex agar lebih siap untuk NLP.

In [ ]:
def clean_text_regex(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", " ", text)
    text = re.sub(r"(?:\+62|62|0)\d{8,13}", " ", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_regex["clean_text_regex"] = df_regex["content"].apply(clean_text_regex)
df_regex["len_before"] = df_regex["content"].str.len()
df_regex["len_after"] = df_regex["clean_text_regex"].str.len()

sample_compare = df_regex[["content", "clean_text_regex", "len_before", "len_after"]].head(10)
sample_compare

In [ ]:
length_stats = pd.DataFrame({
    "metric": ["avg_len_before", "avg_len_after", "median_len_before", "median_len_after"],
    "value": [
        df_regex["len_before"].mean(),
        df_regex["len_after"].mean(),
        df_regex["len_before"].median(),
        df_regex["len_after"].median(),
    ],
})

display(length_stats)

plt.figure(figsize=(10, 4))
sns.kdeplot(df_regex["len_before"], label="Before", fill=True, alpha=0.3)
sns.kdeplot(df_regex["len_after"], label="After", fill=True, alpha=0.3)
plt.title("Distribusi Panjang Teks Sebelum vs Sesudah Regex Cleaning")
plt.xlabel("Panjang Teks")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

### Penjelasan Output
- Kolom before vs after menunjukkan dampak langsung cleaning regex pada teks.
- Rata-rata/median panjang teks setelah cleaning biasanya menurun karena noise dihapus.
- Kurva distribusi memvisualisasikan pergeseran panjang teks secara global.

## 3. Issue Tagging dengan Regex

Mengelompokkan review ke isu utama berbasis aturan regex untuk analisis tematik cepat.

In [ ]:
issue_patterns = {
    "login_otp": r"\blogin\b|\botp\b|verifikasi|kode",
    "error_bug": r"error|gagal|bug|crash|force close|stuck",
    "transaksi": r"transfer|topup|qris|saldo|potong|debit",
    "performa": r"lambat|lemot|loading|cepat",
    "ui_ux": r"ribet|mudah|user friendly|tampilan",
}

def extract_issues(text: str):
    hits = [k for k, p in issue_patterns.items() if re.search(p, text)]
    return hits if hits else ["other"]

df_regex["issue_tags"] = df_regex["clean_text_regex"].apply(extract_issues)
issue_exploded = df_regex[["reviewId", "issue_tags"]].explode("issue_tags")
issue_count = issue_exploded["issue_tags"].value_counts().reset_index()
issue_count.columns = ["issue", "count"]
issue_count

In [ ]:
plt.figure(figsize=(10, 4))
sns.barplot(data=issue_count, x="count", y="issue", palette="crest")
plt.title("Distribusi Issue Tag dari Regex Rules")
plt.xlabel("Jumlah Kemunculan")
plt.ylabel("Issue")
plt.tight_layout()
plt.show()

In [ ]:
if "sentimen_label" not in df_regex.columns and "score" in df_regex.columns:
    def map_sentimen(score):
        if score >= 4:
            return "positif"
        if score == 3:
            return "netral"
        return "negatif"
    df_regex["sentimen_label"] = df_regex["score"].apply(map_sentimen)

issue_sent = issue_exploded.merge(
    df_regex[["reviewId", "sentimen_label"]],
    on="reviewId",
    how="left"
    )

issue_sent_pivot = pd.crosstab(issue_sent["issue_tags"], issue_sent["sentimen_label"])
issue_sent_pct = issue_sent_pivot.div(issue_sent_pivot.sum(axis=1), axis=0).round(3)
display(issue_sent_pivot)
display(issue_sent_pct)

In [ ]:
plt.figure(figsize=(8, 4))
sns.heatmap(issue_sent_pct, annot=True, fmt=".2f", cmap="YlOrRd")
plt.title("Proporsi Sentimen per Issue (Regex Tagging)")
plt.xlabel("Sentimen")
plt.ylabel("Issue")
plt.tight_layout()
plt.show()

### Penjelasan Output
- Bar chart menunjukkan isu regex yang paling banyak muncul.
- Crosstab dan heatmap menunjukkan isu mana yang cenderung dominan pada sentimen negatif.
- Ini membantu prioritas tindakan: isu dengan volume tinggi dan proporsi negatif besar harus ditangani dulu.

## Kesimpulan Regex Analysis

1. Regex efektif untuk mendeteksi noise teks (angka, URL, simbol, karakter berulang).
2. Cleaning regex menurunkan kompleksitas teks dan meningkatkan kesiapan data untuk NLP.
3. Tagging isu berbasis regex memberikan pemetaan masalah yang cepat dan interpretatif.
4. Kombinasi volume isu dan proporsi sentimen negatif dapat dijadikan dasar prioritas perbaikan aplikasi.